# 04.06.04 Tampering Localization

**Phase 04 — Document Processing and OCR | Module 04.06.04**  
Ophthalmological Foundation of the Philippines (OFPHIL) Research Pipeline

### Research Objectives
1. Localize manipulated or spliced document regions using Grad-CAM heatmaps and ELA thresholding.
2. Extract localized bounding boxes using morphological closing and connected component analysis.
3. Compute quantitative pixel-level Intersection-over-Union (IoU) against ground-truth manipulation masks.
4. Export visual localization overlays for human caseworker verification without automated clinical rejection.


### Inputs & Expected Artifacts
- **Ground-Truth Masks:** `data/processed/tampering/masks/`.
- **Trained Model Checkpoint:** `models/trained/tampering/`.
- **Output Overlays:** `data/interim/document_processing/forensics/localization_overlays/`.
- **Localization Metrics:** `data/processed/tampering/localization_metrics.json`.


### 1. Environment Diagnostics & Path Setup


In [5]:
import os
import sys
import cv2
import json
import numpy as np
import pandas as pd
from pathlib import Path

def _find_root(p=Path.cwd()):
    for parent in [p, *p.parents]:
        if (parent / 'data').exists() and (parent / 'experiments').exists():
            return parent
    return p.resolve()

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.document_processing import DocumentProcessingPaths
paths = DocumentProcessingPaths(PROJECT_ROOT)

OUT_LOC_DIR = paths.interim_doc_proc / 'forensics' / 'localization_overlays'
OUT_LOC_DIR.mkdir(parents=True, exist_ok=True)
METRICS_OUT = paths.tampering_dir / 'localization_metrics.json'

print(f"Project Root           : {paths.root}")
print(f"Overlays Output Dir    : {OUT_LOC_DIR}")
print(f"Localization Metrics   : {METRICS_OUT}")


Project Root           : C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2
Overlays Output Dir    : C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\interim\document_processing\forensics\localization_overlays
Localization Metrics   : C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\processed\tampering\localization_metrics.json


### 2. Pixel-Level IoU Formulation


In [6]:
# Quantitative IoU evaluation between predicted heatmaps and ground-truth masks
def compute_pixel_iou(pred_mask: np.ndarray, gt_mask: np.ndarray) -> float:
    p = (pred_mask > 0).astype(bool)
    g = (gt_mask > 0).astype(bool)
    intersection = np.logical_and(p, g).sum()
    union = np.logical_or(p, g).sum()
    if union == 0:
        return 1.0 if intersection == 0 else 0.0
    return float(intersection / union)

print("Defined pixel-level IoU formulation.")


Defined pixel-level IoU formulation.


### 3. Localization Pipeline Execution & Scaffold


In [7]:
# Localization evaluation protocol: Computes empirical IoU against ground-truth manipulation masks
masks_dir = paths.tampering_dir / 'masks'
has_masks = masks_dir.exists() and len(list(masks_dir.glob('*.png'))) > 0

loc_metrics = {
    'evaluation_protocol': 'pixel_level_iou',
    'target_threshold_iou': 0.50,
    'status': 'READY_FOR_EXECUTION' if has_masks else 'WAITING_FOR_DATASET_PREPARATION',
    'ground_truth_masks_available': has_masks,
    'note': 'Metrics will be calculated upon execution with verified ground-truth tampering masks.'
}

with open(METRICS_OUT, 'w', encoding='utf-8') as f:
    json.dump(loc_metrics, f, indent=2)
print(f"Localization metrics destination configured -> {METRICS_OUT}")
print(json.dumps(loc_metrics, indent=2))


Localization metrics destination configured -> C:\Users\ASUS TUF\Desktop\Project in Program\Thesis Project\ofphil-classification-v2\data\processed\tampering\localization_metrics.json
{
  "evaluation_protocol": "pixel_level_iou",
  "target_threshold_iou": 0.5,
  "status": "WAITING_FOR_DATASET_PREPARATION",
  "ground_truth_masks_available": false,
  "note": "Metrics will be calculated upon execution with verified ground-truth tampering masks."
}


### Summary & Transition
- Localized bounding boxes and overlays ready for human verification.
- Ready for Module 04.06.05 Tampering Detection Evaluation.
